# Bloque 2, Tema 3: Ensambles, Stacking y Modelos Híbridos

## Objetivo
Comprender cuándo y cómo combinar múltiples modelos para lograr predicciones más precisas y robustas.

### ¿Por qué combinar modelos?
- **Diversidad**: Diferentes modelos cometen errores distintos
- **Robustez**: Reduce varianza y sesgo simultáneamente
- **Interpretabilidad + Precisión**: Los modelos híbridos balancean ambas


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                             VotingClassifier, BaggingClassifier, AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✅ Librerías cargadas exitosamente")

---
## 1. Diversidad: La Clave de los Ensambles

La regla de oro: **Un ensamble es efectivo SOLO si los modelos cometen errores distintos.**


In [ ]:
# Generar dataset
X, y = make_classification(n_samples=200, n_features=10, n_informative=7, 
                           n_redundant=2, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelos base
rf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)
gb = GradientBoostingClassifier(n_estimators=50, random_state=42, max_depth=3)
lr = LogisticRegression(max_iter=1000, random_state=42)
svm = SVC(kernel='rbf', probability=True, random_state=42)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)
lr.fit(X_train, y_train)
svm.fit(X_train, y_train)

# Predicciones
pred_rf = rf.predict(X_test)
pred_gb = gb.predict(X_test)
pred_lr = lr.predict(X_test)
pred_svm = svm.predict(X_test)

# Matriz de correlación de errores
errors_rf = (pred_rf != y_test).astype(int)
errors_gb = (pred_gb != y_test).astype(int)
errors_lr = (pred_lr != y_test).astype(int)
errors_svm = (pred_svm != y_test).astype(int)

error_corr = np.array([
    [1, np.corrcoef(errors_rf, errors_gb)[0,1], np.corrcoef(errors_rf, errors_lr)[0,1], np.corrcoef(errors_rf, errors_svm)[0,1]],
    [np.corrcoef(errors_gb, errors_rf)[0,1], 1, np.corrcoef(errors_gb, errors_lr)[0,1], np.corrcoef(errors_gb, errors_svm)[0,1]],
    [np.corrcoef(errors_lr, errors_rf)[0,1], np.corrcoef(errors_lr, errors_gb)[0,1], 1, np.corrcoef(errors_lr, errors_svm)[0,1]],
    [np.corrcoef(errors_svm, errors_rf)[0,1], np.corrcoef(errors_svm, errors_gb)[0,1], np.corrcoef(errors_svm, errors_lr)[0,1], 1]
])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(error_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=['RF', 'GB', 'LR', 'SVM'],
            yticklabels=['RF', 'GB', 'LR', 'SVM'],
            cbar_kws={'label': 'Correlación de Errores'}, ax=ax)
ax.set_title('🎯 Matriz de Diversidad\n(Más baja = Más diverso)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Valores cercanos a 0 = Modelos cometen errores distintos (BUENO) ✅")

---
## 2. Voting: Democracia en Acción

Cada modelo vota, gana la clase con más votos.


In [ ]:
voting_hard = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr), ('svm', svm)],
    voting='hard'
)
voting_hard.fit(X_train, y_train)

voting_soft = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr), ('svm', svm)],
    voting='soft'
)
voting_soft.fit(X_train, y_train)

pred_hard = voting_hard.predict(X_test)
pred_soft = voting_soft.predict(X_test)

scores = {
    'RF': accuracy_score(y_test, pred_rf),
    'GB': accuracy_score(y_test, pred_gb),
    'LR': accuracy_score(y_test, pred_lr),
    'SVM': accuracy_score(y_test, pred_svm),
    'Voting Hard': accuracy_score(y_test, pred_hard),
    'Voting Soft': accuracy_score(y_test, pred_soft)
}

fig, ax = plt.subplots(figsize=(10, 6))
models = list(scores.keys())
accs = list(scores.values())
colors = ['#FF6B6B' if 'Voting' not in m else '#4ECDC4' for m in models]
bars = ax.barh(models, accs, color=colors)
ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('🗳️ Voting: Individual vs Ensamble', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1])

for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax.text(acc + 0.02, i, f'{acc:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Voting mejora el rendimiento mediante diversidad")

---
## 3. Bagging: Bootstrapping Aggregating

Random Forest es Bagging con árboles de decisión.


In [ ]:
subsamples = [0.3, 0.5, 0.7, 1.0]
scores_bagging = []

for subsample in subsamples:
    bagging = BaggingClassifier(n_estimators=100, max_samples=subsample, random_state=42)
    bagging.fit(X_train, y_train)
    pred = bagging.predict(X_test)
    score = accuracy_score(y_test, pred)
    scores_bagging.append(score)

n_estimators_list = [10, 30, 50, 100, 200]
scores_n_est = []

for n in n_estimators_list:
    bagging = BaggingClassifier(n_estimators=n, random_state=42)
    bagging.fit(X_train, y_train)
    pred = bagging.predict(X_test)
    score = accuracy_score(y_test, pred)
    scores_n_est.append(score)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(subsamples, scores_bagging, marker='o', linewidth=2, markersize=10, color='#FF6B6B')
axes[0].set_xlabel('Subsample Ratio', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('📦 Bagging: Efecto del Tamaño de Subset', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_estimators_list, scores_n_est, marker='s', linewidth=2, markersize=10, color='#4ECDC4')
axes[1].set_xlabel('Número de Estimadores', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('📦 Bagging: Efecto del Número de Modelos', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Boosting: Aprendiendo de Errores

Secuencia de modelos débiles que se corrigen mutuamente.


In [ ]:
models_comp = {
    'Decision Tree': RandomForestClassifier(n_estimators=1, max_depth=3, random_state=42),
    'AdaBoost (50)': AdaBoostClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
}

scores_comp = {}
for name, model in models_comp.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores_comp[name] = accuracy_score(y_test, pred)

fig, ax = plt.subplots(figsize=(10, 6))
names = list(scores_comp.keys())
accs = list(scores_comp.values())
colors_comp = ['#FF6B6B', '#FFA07A', '#4ECDC4', '#95E1D3']

bars = ax.bar(range(len(names)), accs, color=colors_comp, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('🚀 Boosting vs Bagging vs Individual', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim([0.7, 1.0])

for bar, acc in zip(bars, accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Stacking: Meta-Aprendizaje

⚠️ CRÍTICO: Evitar leakage separando datos para modelos base y meta-modelo.


In [ ]:
# Dividir datos en 3 partes para evitar leakage
X_train_base, X_meta_test, y_train_base, y_meta_test = train_test_split(
    X_train, y_train, test_size=0.4, random_state=42
)
X_train_meta, X_test_meta, y_train_meta, y_test_meta = train_test_split(
    X_meta_test, y_meta_test, test_size=0.5, random_state=42
)

# Entrenar modelos base
base_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42, max_depth=3)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
]

for name, model in base_models:
    model.fit(X_train_base, y_train_base)

# Generar meta-features
meta_features_train = []
for name, model in base_models:
    pred = model.predict_proba(X_train_meta)[:, 1]
    meta_features_train.append(pred)
meta_features_train = np.column_stack(meta_features_train)

meta_features_test = []
for name, model in base_models:
    pred = model.predict_proba(X_test_meta)[:, 1]
    meta_features_test.append(pred)
meta_features_test = np.column_stack(meta_features_test)

# Entrenar meta-modelo
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(meta_features_train, y_train_meta)

pred_stacking = meta_model.predict(meta_features_test)

# Comparar
scores_stack = {
    'RF': accuracy_score(y_test_meta, [m.predict(X_test_meta) for n, m in base_models][0]),
    'GB': accuracy_score(y_test_meta, [m.predict(X_test_meta) for n, m in base_models][1]),
    'LR': accuracy_score(y_test_meta, [m.predict(X_test_meta) for n, m in base_models][2]),
    'SVM': accuracy_score(y_test_meta, [m.predict(X_test_meta) for n, m in base_models][3]),
    'Stacking': accuracy_score(y_test_meta, pred_stacking)
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
names = list(scores_stack.keys())
accs = list(scores_stack.values())
colors_stack = ['#FF6B6B']*4 + ['#4ECDC4']
bars = ax1.bar(names, accs, color=colors_stack, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Accuracy', fontweight='bold', fontsize=12)
ax1.set_title('🎯 Stacking vs Modelos Individuales', fontsize=12, fontweight='bold')
ax1.set_ylim([0.7, 1.0])
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

ax2 = axes[1]
coefficients = meta_model.coef_[0]
model_names_meta = ['RF', 'GB', 'LR', 'SVM']
colors_coef = ['#4ECDC4' if c > 0 else '#FF6B6B' for c in coefficients]
ax2.barh(model_names_meta, coefficients, color=colors_coef, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Coeficiente del Meta-Modelo', fontweight='bold', fontsize=12)
ax2.set_title('🎛️ Ponderación del Meta-Modelo', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

print("El meta-modelo aprendió a dar más peso a los modelos mejores")

---
## 🌟 WOW EXTRA: Análisis de Complementariedad

**¿Cuáles modelos funcionan mejor JUNTOS?**

Cuantificamos qué pares de modelos son más complementarios.


In [ ]:
# Análisis de complementariedad entre pares

models_all = {
    'RF': RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5),
    'GB': GradientBoostingClassifier(n_estimators=50, random_state=42, max_depth=3),
    'LR': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

for name, model in models_all.items():
    model.fit(X_train, y_train)

predictions = {}
for name, model in models_all.items():
    predictions[name] = model.predict(X_test)

# Matriz de complementariedad
model_names = list(predictions.keys())
n_models = len(model_names)
complementarity = np.zeros((n_models, n_models))
voting_gain = np.zeros((n_models, n_models))

for i, model1 in enumerate(model_names):
    for j, model2 in enumerate(model_names):
        if i != j:
            ensemble_pred = np.round((
                (predictions[model1] + predictions[model2]) / 2
            )).astype(int)
            
            acc_individual_avg = (accuracy_score(y_test, predictions[model1]) + 
                                 accuracy_score(y_test, predictions[model2])) / 2
            acc_ensemble = accuracy_score(y_test, ensemble_pred)
            
            voting_gain[i, j] = acc_ensemble - acc_individual_avg
            
            errors1 = (predictions[model1] != y_test).astype(int)
            errors2 = (predictions[model2] != y_test).astype(int)
            correlation_errors = np.corrcoef(errors1, errors2)[0, 1]
            complementarity[i, j] = -correlation_errors

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
sns.heatmap(complementarity, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            xticklabels=model_names, yticklabels=model_names, ax=ax1,
            cbar_kws={'label': 'Complementariedad'})
ax1.set_title('🎯 Matriz de Complementariedad\n(Verde = Mejor juntos)', fontsize=12, fontweight='bold')

ax2 = axes[1]
sns.heatmap(voting_gain, annot=True, fmt='.4f', cmap='RdYlGn', center=0,
            xticklabels=model_names, yticklabels=model_names, ax=ax2,
            cbar_kws={'label': 'Ganancia'})
ax2.set_title('📈 Ganancia de Voting\n(Positivo = Mejora)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Mejor par
best_pair_idx = np.unravel_index(np.argmax(voting_gain), voting_gain.shape)
best_pair = (model_names[best_pair_idx[0]], model_names[best_pair_idx[1]])
best_gain = voting_gain[best_pair_idx]

print(f"\n🏆 Mejor pareja: {best_pair[0]} + {best_pair[1]} (ganancia: {best_gain:.4f})")

---
## Tabla de Decisión: ¿Cuándo usar qué?

| Método | Cuándo usar | Ventaja | Desventaja |
|--------|-----------|---------|-----------|
| **Voting** | Rápido, modelos entrenados | Simple | No aprende combinación |
| **Bagging** | Alta varianza (árboles) | Reduce varianza | No reduce bias |
| **Boosting** | Alto sesgo, máxima precisión | Reduce bias | Complejo, overfitting |
| **Stacking** | Máxima precisión, datos abundantes | Óptimo | Computacionalmente caro |

### Recomendaciones
- **Producción (latencia baja)** → Voting / Bagging
- **Máxima precisión** → Stacking / Boosting
- **Datos complejos** → Ensambles + Híbridos


In [ ]:
print("\n" + "="*70)
print("CLAVES DEL ÉXITO CON ENSAMBLES:")
print("="*70)
print("\n1️⃣  DIVERSIDAD: Modelos que cometan errores diferentes")
print("2️⃣  NO OVERFITTING: Regulariza meta-modelos")
print("3️⃣  BALANCE: Más complejo no siempre es mejor")
print("4️⃣  VALIDACIÓN: Evita leakage en Stacking")
print("5️⃣  INTERPRETABILIDAD: Considera trade-offs")
print("\n" + "="*70)